In [1]:
import json
import os
from pathlib import Path

import requests
from dotenv import load_dotenv

In [6]:
# Parameters for API request
START_DATE = "2023-01-01"
END_DATE = "2023-12-31"
PAGE_SIZE = 40
MAX_PAGES = 2
ORDERING = "-added"

In [7]:
# Cleaned date strings for file naming
START_DATE_CLEAN = START_DATE.replace("-", "")
END_DATE_CLEAN = END_DATE.replace("-", "")

In [10]:
load_dotenv()

API_KEY = os.getenv("RAWG_API_KEY")

if not API_KEY:
    raise ValueError("RAWG_API_KEY not found in .env")

url = "https://api.rawg.io/api/games"

params = {
        "key": API_KEY,
        "dates": f"{START_DATE},{END_DATE}",
        "ordering": ORDERING,
        "page_size": PAGE_SIZE,
    }

# Info for validation
total_games = 0
pages_processed = 0
page = 1

In [9]:
while url and pages_processed < MAX_PAGES:
    response = requests.get(url, params=params, timeout=30)
    response.raise_for_status()

    data = response.json()

    output_dir = Path("data/raw")
    output_dir.mkdir(parents=True, exist_ok=True)

    output_file = output_dir / f"rawg_games_test_{START_DATE_CLEAN}_{END_DATE_CLEAN}_{page:03}.json"
    with output_file.open("w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

    total_games += len(data.get('results', []))
    pages_processed += 1
    page += 1

    print(f"Status code: {response.status_code}")
    print(f"Count field: {data.get('count')}")
    print(f"Results returned: {len(data.get('results', []))}")
    print(f"Saved to: {output_file}")

    url = data["next"]
    params = None
    
print(f"Total games processed: {total_games}")
print(f"Pages processed: {pages_processed}")

Status code: 200
Count field: 59122
Results returned: 40
Saved to: data/raw/rawg_games_test_20230101_20231231_001.json
Status code: 200
Count field: 59122
Results returned: 40
Saved to: data/raw/rawg_games_test_20230101_20231231_002.json
Total games processed: 80
Pages processed: 2


In [20]:
page = 0
while page == 0:
    response = requests.get(url, params=params, timeout=30)
    response.raise_for_status()

    data = response.json()

    print(f"Count field: {data.get('count')}")
    
    page += 1

print("OK")

Count field: 17925
OK
